# 02 – Mô hình baseline (Sleep DSS)

**Bài toán:** phân loại nhị phân. Từ thói quen buổi tối, ước lượng xác suất bị *nợ ngủ đáng kể* (`Moderate Debt` hoặc `Severe Sleep Debt` = 1; `Optimal Recovery`, `Mild Deficit` = 0). Cách gộp nhãn có thể đổi ở biến `POSITIVE` bên dưới sau khi nhóm chốt.

**Chỉ dùng đặc trưng hành vi/ngữ cảnh.** Các cột kết quả giấc ngủ (`total_sleep_hours`, `sleep_latency_min`, `deep_sleep_pct`, `rem_sleep_pct`, `morning_alarm_snoozes`, `next_day_fatigue_score`) bị loại để tránh rò rỉ dữ liệu.

Thứ tự: baseline đoán lớp phổ biến nhất → Logistic Regression → lưu `models/model.joblib`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

RANDOM_STATE = 42
DATA = Path("../data/raw")
MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)

df = pd.read_csv(next(DATA.glob("*.csv")))
df.shape

In [ ]:
LABEL = "sleep_debt_category"
POSITIVE = ["Moderate Debt", "Severe Sleep Debt"]   # nhãn 1 = nợ ngủ đáng kể

NUM = ["age", "bedtime_phone_minutes", "screen_brightness_pct",
       "blue_light_filter_active", "caffeine_post_5pm_mg", "physical_activity_min"]
CAT = ["gender", "occupation_type", "chronotype", "primary_bedtime_app"]
FEATURES = NUM + CAT

X = df[FEATURES]
y = df[LABEL].isin(POSITIVE).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)
print("Tỉ lệ nhãn 1:", y.mean().round(3))

In [ ]:
def make_pipe(model):
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])
    return Pipeline([("pre", pre), ("clf", model)])

def evaluate(name, pipe):
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    return {"model": name,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba)}

In [ ]:
# Baseline 1: luôn đoán lớp phổ biến nhất
dummy = make_pipe(DummyClassifier(strategy="most_frequent")).fit(X_train, y_train)

# Baseline 2: Logistic Regression
logreg = make_pipe(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)).fit(X_train, y_train)

results = pd.DataFrame([evaluate("Most frequent", dummy),
                        evaluate("Logistic Regression", logreg)]).set_index("model").round(3)
results

In [ ]:
pred = logreg.predict(X_test)
pd.DataFrame(confusion_matrix(y_test, pred), index=["true 0", "true 1"], columns=["pred 0", "pred 1"])

In [ ]:
# Lưu bản đầu để Chính tích hợp: cả pipeline (tiền xử lý + mô hình).
# Đầu vào: DataFrame thô có đúng các cột trong FEATURES.
# predict_proba(X)[:, 1] = xác suất nợ ngủ đáng kể.
joblib.dump(logreg, MODELS / "model.joblib")

m = joblib.load(MODELS / "model.joblib")
proba = m.predict_proba(X_test.head(5))[:, 1]
print(m.classes_, proba.round(3))
assert ((proba >= 0) & (proba <= 1)).all()